# Transcripción de Clases de Teología

Usa Whisper large-v3 con aceleración Metal (Apple Silicon) para transcribir audios de clase a texto.

In [1]:
# Instalar dependencias (solo la primera vez)
!pip install mlx-whisper mutagen

In [2]:
import mlx_whisper
from pathlib import Path
from mutagen import File as MutagenFile
import time

MODEL = "mlx-community/whisper-large-v3-turbo"

print(f"Modelo: {MODEL}")
print("Aceleración: Apple Metal (GPU)")

/Users/delpiwma/Documents/Projects/teologia/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelo: mlx-community/whisper-large-v3-turbo
Aceleración: Apple Metal (GPU)


In [3]:
# Configuración
AUDIO_DIR = Path("./audios")
OUTPUT_DIR = Path("./transcripciones")
OUTPUT_DIR.mkdir(exist_ok=True)

EXTENSIONS = {".mp3", ".m4a", ".wav", ".ogg", ".flac", ".mp4", ".webm"}

audios = sorted([f for f in AUDIO_DIR.iterdir() if f.suffix.lower() in EXTENSIONS])

def get_duration(path):
    """Obtener duración del audio en segundos."""
    try:
        audio = MutagenFile(path)
        return audio.info.length if audio and audio.info else 0
    except:
        return 0

def fmt_duration(seconds):
    """Formatear segundos a HH:MM:SS."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h}h {m:02d}m {s:02d}s"
    return f"{m}m {s:02d}s"

# Info de archivos
total_duration = 0
print(f"{'#':<4} {'Archivo':<40} {'Duración':<12} {'Estado'}")
print("─" * 70)
for idx, a in enumerate(audios, 1):
    dur = get_duration(a)
    total_duration += dur
    exists = "✅ ya transcrito" if (OUTPUT_DIR / f"{a.stem}.txt").exists() else "⏳ pendiente"
    print(f"{idx:<4} {a.name:<40} {fmt_duration(dur):<12} {exists}")

print("─" * 70)
print(f"Total: {len(audios)} archivos | {fmt_duration(total_duration)} de audio")

#    Archivo                                  Duración     Estado
──────────────────────────────────────────────────────────────────────
1    2026-04-24 18-37-15.m4a                  1h 57m 39s   ⏳ pendiente
2    2026-05-08 18-34-21.m4a                  1h 59m 31s   ⏳ pendiente
──────────────────────────────────────────────────────────────────────
Total: 2 archivos | 3h 57m 10s de audio


In [4]:
# Transcribir con progreso
pending = [(a, get_duration(a)) for a in audios if not (OUTPUT_DIR / f"{a.stem}.txt").exists()]

if not pending:
    print("✅ Todos los audios ya están transcritos.")
else:
    total_pending_dur = sum(d for _, d in pending)
    elapsed_total = 0
    
    print(f"\n🎙️  Transcribiendo {len(pending)} archivos ({fmt_duration(total_pending_dur)})\n")
    
    for idx, (audio_path, duration) in enumerate(pending, 1):
        output_path = OUTPUT_DIR / f"{audio_path.stem}.txt"
        
        # Progreso
        pct = (idx - 1) / len(pending) * 100
        bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
        
        # ETA
        if elapsed_total > 0 and idx > 1:
            avg_ratio = elapsed_total / sum(d for _, d in pending[:idx-1])
            remaining_dur = sum(d for _, d in pending[idx-1:])
            eta = fmt_duration(remaining_dur * avg_ratio)
        else:
            eta = "calculando..."
        
        print(f"[{bar}] {pct:.0f}% | ETA: {eta}")
        print(f"  → {audio_path.name} ({fmt_duration(duration)})")
        
        start = time.time()
        result = mlx_whisper.transcribe(
            str(audio_path),
            path_or_hf_repo=MODEL,
            language="es",
        )
        took = time.time() - start
        elapsed_total += took
        
        output_path.write_text(result["text"], encoding="utf-8")
        speed = duration / took if took > 0 else 0
        print(f"  ✅ Listo en {fmt_duration(took)} ({speed:.1f}x realtime) | {len(result['text'])} chars\n")
    
    # Final
    bar = "█" * 20
    print(f"[{bar}] 100%")
    print(f"\n🏁 Completo. {len(pending)} archivos en {fmt_duration(elapsed_total)}")
    print(f"   Velocidad promedio: {total_pending_dur / elapsed_total:.1f}x realtime")


🎙️  Transcribiendo 2 archivos (3h 57m 10s)

[░░░░░░░░░░░░░░░░░░░░] 0% | ETA: calculando...
  → 2026-04-24 18-37-15.m4a (1h 57m 39s)


Fetching 4 files: 100%|██████████| 4/4 [00:28<00:00,  7.23s/it]


  ✅ Listo en 4m 09s (28.3x realtime) | 101166 chars

[██████████░░░░░░░░░░] 50% | ETA: 4m 13s
  → 2026-05-08 18-34-21.m4a (1h 59m 31s)
  ✅ Listo en 3m 44s (31.9x realtime) | 101080 chars

[████████████████████] 100%

🏁 Completo. 2 archivos en 7m 54s
   Velocidad promedio: 30.0x realtime
